In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog, scrolledtext
import pandas as pd
import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import threading
from datetime import datetime
import json
import os
import matplotlib.pyplot as plt
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

# CatBoost implementation with MOWCA optimization
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import time

class MOWCA:
    """Multi-Objective Water Cycle Algorithm for hyperparameter optimization (Optimized)"""
    def __init__(self, objective_func, n_vars=8, lb=None, ub=None, n_pop=15, 
                 n_iter=15, n_streams=8, n_sea=3, nsr=2, dmax=1e-5, 
                 verbose=False):
        self.objective_func = objective_func
        self.n_vars = n_vars
        self.lb = lb if lb is not None else [1, 0.01, 50, 0.01, 100, 0.5, 0.5, 1]
        self.ub = ub if ub is not None else [10, 0.3, 500, 10, 255, 1.0, 1.0, 10]
        self.n_pop = n_pop
        self.n_iter = n_iter
        self.n_streams = n_streams
        self.n_sea = n_sea
        self.nsr = nsr
        self.dmax = dmax
        self.verbose = verbose
        self.best_solution = None
        self.best_fitness = None
        self.history = []
        
    def initialize_population(self):
        population = np.random.uniform(
            low=self.lb, 
            high=self.ub, 
            size=(self.n_pop, self.n_vars)
        )
        return population
    
    def fitness(self, solution):
        return self.objective_func(solution)
    
    def run(self):
        # Initialize population
        population = self.initialize_population()
        fitness = np.array([self.fitness(ind) for ind in population])
        
        # Initialize seas and streams
        sorted_idx = np.argsort(fitness)
        sea_idx = sorted_idx[:self.n_sea]
        stream_idx = sorted_idx[self.n_sea:]
        
        # Main loop
        for iteration in range(self.n_iter):
            # Update streams
            for i in stream_idx:
                # Determine nearest sea
                sea_distances = [np.linalg.norm(population[i] - population[j]) 
                               for j in sea_idx]
                nearest_sea_idx = sea_idx[np.argmin(sea_distances)]
                
                # Update stream position (reduced step size for faster convergence)
                step = np.random.uniform(0, 1.5) * 0.01
                population[i] += step * np.random.randn(self.n_vars) * \
                                 (population[nearest_sea_idx] - population[i])
                
                # Boundary check
                population[i] = np.clip(population[i], self.lb, self.ub)
                fitness[i] = self.fitness(population[i])
            
            # Update seas
            for i in sea_idx:
                step = np.random.uniform(0, 1.5) * 0.01
                population[i] += step * np.random.randn(self.n_vars) * \
                                 (population[sea_idx[np.argmin(fitness[sea_idx])]] - population[i])
                population[i] = np.clip(population[i], self.lb, self.ub)
                fitness[i] = self.fitness(population[i])
            
            # Evaporation and raining process (reduced probability)
            for i in sea_idx[1:]:
                if np.random.rand() < 0.05:  # Reduced evaporation probability
                    population[i] = np.random.uniform(self.lb, self.ub)
                    fitness[i] = self.fitness(population[i])
            
            # Update best solution
            best_idx = np.argmin(fitness)
            if self.best_fitness is None or fitness[best_idx] < self.best_fitness:
                self.best_fitness = fitness[best_idx]
                self.best_solution = population[best_idx].copy()
            
            self.history.append(self.best_fitness)
            
            if self.verbose and iteration % 5 == 0:
                print(f"Iteration {iteration}: Best Fitness = {self.best_fitness:.4f}")
        
        return self.best_solution, self.best_fitness

class CatBoostWithMOWCA:
    """CatBoost Regressor with MOWCA hyperparameter optimization (Optimized)"""
    def __init__(self):
        self.model = None
        self.best_params = None
        self.scaler = StandardScaler()
        self.r2_score = None
        self.rmse = None
        self.mae = None
        self.cv_scores = None
        self.feature_importance = None
        
    def optimize_hyperparameters(self, X, y, n_folds=5):
        """Optimize hyperparameters using MOWCA with reduced CV folds"""
        
        # Use a subset of data for faster optimization
        X_subset, _, y_subset, _ = train_test_split(
            X, y, train_size=0.5, random_state=42
        )
        
        def objective(params):
            # Decode parameters
            depth = int(params[0])
            learning_rate = float(params[1])
            iterations = int(params[2])
            l2_leaf_reg = float(params[3])
            border_count = int(params[4])
            subsample = float(params[5])
            colsample = float(params[6])
            min_data_in_leaf = int(params[7])
            
            # Create CatBoost model with optimized settings
            model = CatBoostRegressor(
                depth=depth,
                learning_rate=learning_rate,
                iterations=iterations,
                l2_leaf_reg=l2_leaf_reg,
                border_count=border_count,
                subsample=subsample,
                colsample_bylevel=colsample,
                min_data_in_leaf=min_data_in_leaf,
                loss_function='RMSE',
                eval_metric='R2',
                random_seed=42,
                verbose=False,
                allow_writing_files=False,
                thread_count=-1  # Use all CPU cores
            )
            
            # Use reduced CV folds for faster optimization
            kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
            cv_scores = cross_val_score(
                model, X_subset, y_subset, cv=kf, 
                scoring='r2', n_jobs=-1
            )
            
            # Return negative R2 for minimization
            return -np.mean(cv_scores)
        
        # Reduced parameter bounds for faster convergence
        lb = [2, 0.05, 100, 0.1, 100, 0.6, 0.6, 1]
        ub = [8, 0.2, 400, 5, 200, 0.9, 0.9, 5]
        
        # MOWCA optimization with reduced population and iterations
        mowca = MOWCA(
            objective_func=objective,
            n_vars=8,
            lb=lb,
            ub=ub,
            n_pop=15,  # Reduced population
            n_iter=12,  # Reduced iterations
            n_streams=8,
            n_sea=3,
            verbose=True
        )
        
        print("⏳ Running MOWCA optimization (this may take 2-5 minutes)...")
        start_time = time.time()
        best_params, best_fitness = mowca.run()
        elapsed = time.time() - start_time
        print(f"⏱️ Optimization completed in {elapsed/60:.1f} minutes")
        
        # Decode best parameters
        self.best_params = {
            'depth': int(best_params[0]),
            'learning_rate': float(best_params[1]),
            'iterations': int(best_params[2]),
            'l2_leaf_reg': float(best_params[3]),
            'border_count': int(best_params[4]),
            'subsample': float(best_params[5]),
            'colsample_bylevel': float(best_params[6]),
            'min_data_in_leaf': int(best_params[7])
        }
        
        print("\n✅ BEST PARAMETERS FOUND (MOWCA + 5-fold CV):")
        print("-" * 50)
        for key, value in self.best_params.items():
            print(f"  {key}: {value}")
        print(f"  Best CV R²: {-best_fitness:.4f}")
        print("-" * 50)
        
        return self.best_params
    
    def train(self, X, y, params=None):
        """Train CatBoost model with optimal parameters"""
        if params is None:
            params = self.best_params
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        # Create and train model with optimized settings
        self.model = CatBoostRegressor(
            depth=params['depth'],
            learning_rate=params['learning_rate'],
            iterations=params['iterations'],
            l2_leaf_reg=params['l2_leaf_reg'],
            border_count=params['border_count'],
            subsample=params['subsample'],
            colsample_bylevel=params['colsample_bylevel'],
            min_data_in_leaf=params['min_data_in_leaf'],
            loss_function='RMSE',
            eval_metric='R2',
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1  # Use all CPU cores
        )
        
        # Train model
        print("⏳ Training final CatBoost model...")
        self.model.fit(X_scaled, y, verbose=False)
        
        # Calculate metrics on training data
        predictions = self.model.predict(X_scaled)
        self.r2_score = r2_score(y, predictions)
        self.rmse = np.sqrt(mean_squared_error(y, predictions))
        self.mae = mean_absolute_error(y, predictions)
        
        # Get feature importance
        self.feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': self.model.feature_importances_ / 100
        }).sort_values('importance', ascending=False)
        
        return self.model

class ConcreteStrengthPredictor:
    def __init__(self):
        self.setup_style()
        self.setup_paths()
        self.load_data()
        self.train_model()
        self.create_gui()
        
    def setup_style(self):
        self.colors = {
            'primary': '#2E86AB',
            'secondary': '#A23B72',
            'accent': '#F18F01',
            'background': '#F7F9FC',
            'card_bg': '#FFFFFF',
            'success': '#28A745',
            'warning': '#FFC107',
            'danger': '#DC3545',
            'steel_blue': '#4682B4',
            'concrete_gray': '#696969',
        }
        
    def setup_paths(self):
        self.data_path = r"D:\2026 Work\My Papers\GGSB\Data\1-s2.0-S2214509525006989-mmc1.csv"
        self.save_dir = r"D:\2026 Work\My Papers\GGSB\GUI Application"
        os.makedirs(self.save_dir, exist_ok=True)
        self.history_file = os.path.join(self.save_dir, "prediction_history.json")
        
    def load_data(self):
        try:
            print(f"📂 Loading data from: {self.data_path}")
            
            # Try different encodings
            encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252', 'utf-16']
            df = None
            
            for encoding in encodings:
                try:
                    df = pd.read_csv(self.data_path, encoding=encoding)
                    print(f"✅ Successfully loaded with encoding: {encoding}")
                    break
                except UnicodeDecodeError:
                    continue
                except Exception as e:
                    print(f"⚠️ Failed with {encoding}: {str(e)[:50]}")
                    continue
            
            if df is None:
                # Try with engine='python' as fallback
                df = pd.read_csv(self.data_path, encoding='latin1', engine='python')
                print("✅ Loaded with latin1 encoding using python engine")
            
            self.df = df
            print(f"✅ Dataset loaded successfully!")
            print(f"📐 Dataset shape: {self.df.shape}")
            print(f"📋 Column names: {self.df.columns.tolist()}")
            
            # Clean column names
            self.df.columns = self.df.columns.str.strip()
            
            # Find target column (C-S or CS or MPa)
            target_col = None
            for col in self.df.columns:
                col_lower = col.lower()
                if 'c-s' in col_lower or 'cs' in col_lower or 'mpa' in col_lower or 'strength' in col_lower:
                    target_col = col
                    break
            
            if target_col is None:
                # Try to find numeric column that could be target
                numeric_cols = self.df.select_dtypes(include=[np.number]).columns
                if len(numeric_cols) > 0:
                    # Use the last numeric column as target
                    target_col = numeric_cols[-1]
                    print(f"⚠️ Using last numeric column as target: {target_col}")
                else:
                    target_col = self.df.columns[-1]
            
            self.target_name = target_col
            print(f"🎯 Target column: '{self.target_name}'")
            
            # Prepare features and target
            self.X = self.df.drop(columns=[target_col])
            self.y = self.df[target_col]
            self.feature_names = self.X.columns.tolist()
            
            # Clean feature names (remove special characters)
            self.feature_names = [f.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_') for f in self.feature_names]
            self.X.columns = self.feature_names
            
            # Calculate statistics for each feature
            self.feature_stats = {}
            for feature in self.feature_names:
                self.feature_stats[feature] = {
                    'min': self.X[feature].min(),
                    'max': self.X[feature].max(),
                    'mean': self.X[feature].mean(),
                    'std': self.X[feature].std(),
                    'median': self.X[feature].median(),
                    'q1': self.X[feature].quantile(0.25),
                    'q3': self.X[feature].quantile(0.75)
                }
            
            print(f"🔍 Features loaded: {len(self.feature_names)}")
            print(f"📊 Features: {', '.join(self.feature_names[:10])}{'...' if len(self.feature_names) > 10 else ''}")
            
            # Display target statistics
            print(f"\n📊 Target Statistics ({self.target_name}):")
            print(f"   • Mean: {self.y.mean():.2f} MPa")
            print(f"   • Std: {self.y.std():.2f} MPa")
            print(f"   • Min: {self.y.min():.2f} MPa")
            print(f"   • Max: {self.y.max():.2f} MPa")
            
        except Exception as e:
            error_msg = f"Failed to load data: {str(e)}"
            print(f"❌ {error_msg}")
            import traceback
            traceback.print_exc()
            messagebox.showerror("Data Loading Error", error_msg)
            raise
    
    def train_model(self):
        try:
            print("\n🤖 Optimizing CatBoost with MOWCA + 5-fold CV...")
            
            # Check if we can load pre-trained model
            model_path = os.path.join(self.save_dir, "catboost_model.cbm")
            params_path = os.path.join(self.save_dir, "best_params.json")
            
            if os.path.exists(model_path) and os.path.exists(params_path):
                try:
                    print("⏳ Loading pre-trained model...")
                    with open(params_path, 'r') as f:
                        self.best_params = json.load(f)
                    
                    self.model = CatBoostWithMOWCA()
                    self.model.best_params = self.best_params
                    self.model.scaler.fit(self.X)
                    self.model.model = CatBoostRegressor()
                    self.model.model.load_model(model_path)
                    
                    # Calculate metrics
                    X_scaled = self.model.scaler.transform(self.X)
                    predictions = self.model.model.predict(X_scaled)
                    self.model.r2_score = r2_score(self.y, predictions)
                    self.model.rmse = np.sqrt(mean_squared_error(self.y, predictions))
                    self.model.mae = mean_absolute_error(self.y, predictions)
                    
                    # Get feature importance
                    self.model.feature_importance = pd.DataFrame({
                        'feature': self.X.columns,
                        'importance': self.model.model.feature_importances_ / 100
                    }).sort_values('importance', ascending=False)
                    
                    print("✅ Pre-trained model loaded successfully!")
                    print(f"\n📈 Model Performance (Full Dataset):")
                    print(f"   • R² Score: {self.model.r2_score:.4f}")
                    print(f"   • RMSE: {self.model.rmse:.2f} MPa")
                    print(f"   • MAE: {self.model.mae:.2f} MPa")
                    
                    # Display feature importance
                    print(f"\n🏆 Top 5 Most Important Features:")
                    for i, (_, row) in enumerate(self.model.feature_importance.head(5).iterrows(), 1):
                        print(f"   {i}. {row['feature']:30} : {row['importance']:.4f}")
                    return
                except Exception as e:
                    print(f"⚠️ Could not load pre-trained model: {e}")
                    print("🔄 Training new model...")
            
            # Optimize hyperparameters
            optimizer = CatBoostWithMOWCA()
            best_params = optimizer.optimize_hyperparameters(
                self.X, self.y, n_folds=5
            )
            
            # Save best parameters
            with open(params_path, 'w') as f:
                json.dump(best_params, f)
            
            # Train final model
            print("\n🎯 Training final CatBoost model...")
            self.model = optimizer
            self.model.train(self.X, self.y, best_params)
            
            # Save model
            self.model.model.save_model(model_path)
            print(f"✅ Model saved to: {model_path}")
            
            print("\n✅ Model training completed!")
            print(f"\n📈 Model Performance (Full Dataset):")
            print(f"   • R² Score: {self.model.r2_score:.4f}")
            print(f"   • RMSE: {self.model.rmse:.2f} MPa")
            print(f"   • MAE: {self.model.mae:.2f} MPa")
            
            # Display feature importance
            print(f"\n🏆 Top 5 Most Important Features:")
            for i, (_, row) in enumerate(self.model.feature_importance.head(5).iterrows(), 1):
                print(f"   {i}. {row['feature']:30} : {row['importance']:.4f}")
            
        except Exception as e:
            error_msg = f"Failed to train model: {str(e)}"
            print(f"❌ {error_msg}")
            import traceback
            traceback.print_exc()
            messagebox.showerror("Model Training Error", error_msg)
            raise
    
    def create_gui(self):
        self.root = tk.Tk()
        self.root.title("Concrete C-S (MPa) Predictor - CatBoost + MOWCA")
        self.root.geometry("1400x900")
        self.root.configure(bg=self.colors['background'])
        
        # Center window
        self.root.update_idletasks()
        width = self.root.winfo_width()
        height = self.root.winfo_height()
        x = (self.root.winfo_screenwidth() // 2) - (width // 2)
        y = (self.root.winfo_screenheight() // 2) - (height // 2)
        self.root.geometry(f'1400x900+{x}+{y}')
        
        # Create notebook for tabs
        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill='both', expand=True, padx=10, pady=10)
        
        # Create tabs
        self.prediction_tab = self.create_prediction_tab()
        self.analysis_tab = self.create_analysis_tab()
        self.history_tab = self.create_history_tab()
        
        # Add tabs to notebook
        self.notebook.add(self.prediction_tab, text="📊 Prediction")
        self.notebook.add(self.analysis_tab, text="📈 Analysis")
        self.notebook.add(self.history_tab, text="📋 History")
        
        # Initialize prediction history
        self.prediction_history = []
        self.load_history()
        
        print("\n✅ GUI created successfully!")
    
    def create_prediction_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Main container
        main_container = ttk.Frame(tab)
        main_container.pack(fill='both', expand=True, padx=20, pady=20)
        
        # Left panel - Inputs
        left_panel = ttk.LabelFrame(main_container, text="Input Parameters", padding=15)
        left_panel.pack(side='left', fill='both', expand=True, padx=(0, 10))
        
        # Right panel - Results
        right_panel = ttk.LabelFrame(main_container, text="Prediction Results", padding=15)
        right_panel.pack(side='right', fill='both', expand=True, padx=(10, 0))
        
        # ========== LEFT PANEL: Input Parameters ==========
        
        # Create canvas with scrollbar for inputs
        input_canvas = tk.Canvas(left_panel)
        input_scrollbar = ttk.Scrollbar(left_panel, orient="vertical", command=input_canvas.yview)
        input_frame = ttk.Frame(input_canvas)
        
        input_frame.bind("<Configure>", lambda e: input_canvas.configure(scrollregion=input_canvas.bbox("all")))
        input_canvas.create_window((0, 0), window=input_frame, anchor="nw")
        input_canvas.configure(yscrollcommand=input_scrollbar.set)
        
        input_canvas.pack(side="left", fill="both", expand=True)
        input_scrollbar.pack(side="right", fill="y")
        
        # Quick selection frame
        quick_frame = ttk.LabelFrame(input_frame, text="Quick Selections", padding=10)
        quick_frame.pack(fill='x', pady=(0, 15))
        
        # Preset selection
        ttk.Label(quick_frame, text="Select Preset:", font=("Arial", 10)).pack(anchor='w')
        
        self.preset_var = tk.StringVar(value="Standard Mix")
        preset_combo = ttk.Combobox(quick_frame, textvariable=self.preset_var, 
                                   values=["Standard Mix", "High Strength Mix", 
                                           "Low Strength Mix", "Optimal Mix", "Custom"],
                                   state="readonly", width=25)
        preset_combo.pack(fill='x', pady=(5, 10))
        preset_combo.bind("<<ComboboxSelected>>", self.load_preset)
        
        # Preset buttons
        preset_buttons_frame = ttk.Frame(quick_frame)
        preset_buttons_frame.pack(fill='x')
        
        ttk.Button(preset_buttons_frame, text="Low Strength", 
                  command=lambda: self.load_preset_by_name("Low Strength Mix")).pack(side='left', padx=2)
        ttk.Button(preset_buttons_frame, text="Medium Strength", 
                  command=lambda: self.load_preset_by_name("Standard Mix")).pack(side='left', padx=2)
        ttk.Button(preset_buttons_frame, text="High Strength", 
                  command=lambda: self.load_preset_by_name("High Strength Mix")).pack(side='left', padx=2)
        
        # Random buttons
        random_frame = ttk.Frame(quick_frame)
        random_frame.pack(fill='x', pady=(10, 0))
        
        ttk.Button(random_frame, text="🎲 Random Sample", 
                  command=self.fill_random_sample).pack(side='left', padx=2)
        ttk.Button(random_frame, text="📊 Best Case", 
                  command=self.fill_best_case).pack(side='left', padx=2)
        ttk.Button(random_frame, text="📉 Worst Case", 
                  command=self.fill_worst_case).pack(side='left', padx=2)
        
        # Feature inputs (all features)
        input_features_frame = ttk.LabelFrame(input_frame, text="Feature Values", padding=10)
        input_features_frame.pack(fill='both', expand=True)
        
        self.entries = {}
        
        # Create input fields for all features
        for i, feature in enumerate(self.feature_names):
            feat_frame = ttk.Frame(input_features_frame)
            feat_frame.pack(fill='x', pady=5)
            
            stats = self.feature_stats[feature]
            
            # Label with importance indicator
            importance = self.model.feature_importance[self.model.feature_importance['feature'] == feature]['importance'].values[0] if feature in self.model.feature_importance['feature'].values else 0
            importance_star = " ⭐" if importance > 0.1 else " ✦" if importance > 0.05 else ""
            
            label = tk.Label(feat_frame, 
                            text=f"{feature}{importance_star}:",
                            font=("Arial", 9, "bold"),
                            fg=self.get_importance_color(importance),
                            anchor='w',
                            width=25)
            label.pack(side='left')
            
            # Entry with range info
            entry_frame = ttk.Frame(feat_frame)
            entry_frame.pack(side='left', padx=5)
            
            entry = ttk.Entry(entry_frame, width=15, font=("Arial", 9))
            entry.insert(0, f"{stats['mean']:.2f}")
            entry.pack(side='left')
            
            # Range label
            range_label = tk.Label(entry_frame, 
                                  text=f"[{stats['min']:.1f}-{stats['max']:.1f}]",
                                  font=("Arial", 8),
                                  fg=self.colors['concrete_gray'])
            range_label.pack(side='left', padx=5)
            
            self.entries[feature] = entry
            
            # Slider for easy adjustment
            if stats['max'] - stats['min'] > 0:
                slider_frame = ttk.Frame(feat_frame)
                slider_frame.pack(side='left', padx=10)
                
                slider = ttk.Scale(slider_frame, 
                                  from_=stats['min'], 
                                  to=stats['max'],
                                  value=stats['mean'],
                                  orient='horizontal',
                                  length=150,
                                  command=lambda v, f=feature: self.update_entry_from_slider(f, float(v)))
                slider.pack()
        
        # Control buttons at bottom of left panel
        control_frame = ttk.Frame(input_features_frame)
        control_frame.pack(fill='x', pady=(15, 5))
        
        self.predict_btn = ttk.Button(control_frame,
                                     text="🔮 PREDICT C-S (MPa)",
                                     command=self.threaded_predict,
                                     style='Accent.TButton')
        self.predict_btn.pack(side='left', padx=5, ipadx=10, ipady=8)
        
        ttk.Button(control_frame, text="🗑️ Clear All",
                  command=self.clear_inputs).pack(side='left', padx=5, ipady=8)
        
        # ========== RIGHT PANEL: Results ==========
        
        # Prediction result display
        result_display_frame = ttk.Frame(right_panel)
        result_display_frame.pack(fill='x', pady=(0, 15))
        
        self.result_var = tk.StringVar(value="Ready for prediction")
        result_label = tk.Label(result_display_frame,
                               textvariable=self.result_var,
                               font=("Arial", 48, "bold"),
                               fg=self.colors['steel_blue'])
        result_label.pack()
        
        # Unit label
        unit_label = tk.Label(result_display_frame,
                             text="Megapascals (MPa)",
                             font=("Arial", 14),
                             fg=self.colors['concrete_gray'])
        unit_label.pack()
        
        # Detailed info frame
        detail_frame = ttk.LabelFrame(right_panel, text="Prediction Details", padding=15)
        detail_frame.pack(fill='both', expand=True, pady=(10, 0))
        
        # Create notebook for detailed results
        detail_notebook = ttk.Notebook(detail_frame)
        detail_notebook.pack(fill='both', expand=True)
        
        # Tab 1: Feature Contributions
        contributions_tab = ttk.Frame(detail_notebook)
        detail_notebook.add(contributions_tab, text="Feature Contributions")
        
        self.contributions_text = scrolledtext.ScrolledText(contributions_tab,
                                                          height=10,
                                                          font=("Arial", 9),
                                                          wrap=tk.WORD)
        self.contributions_text.pack(fill='both', expand=True)
        self.contributions_text.insert(1.0, "Feature contributions will appear here after prediction.")
        self.contributions_text.config(state='disabled')
        
        # Tab 2: Statistics
        stats_tab = ttk.Frame(detail_notebook)
        detail_notebook.add(stats_tab, text="Statistics")
        
        stats_text = scrolledtext.ScrolledText(stats_tab,
                                             height=10,
                                             font=("Arial", 9),
                                             wrap=tk.WORD)
        stats_text.pack(fill='both', expand=True)
        
        # Add model statistics
        stats_info = f"""MODEL PERFORMANCE (CatBoost + MOWCA):
• R² Score: {self.model.r2_score:.4f}
• RMSE: {self.model.rmse:.2f} MPa
• MAE: {self.model.mae:.2f} MPa
• Training Samples: {len(self.X)}
• Features Used: {len(self.feature_names)}

OPTIMAL HYPERPARAMETERS (MOWCA + 5-fold CV):
• Depth: {self.model.best_params['depth']}
• Learning Rate: {self.model.best_params['learning_rate']:.4f}
• Iterations: {self.model.best_params['iterations']}
• L2 Leaf Reg: {self.model.best_params['l2_leaf_reg']:.4f}
• Border Count: {self.model.best_params['border_count']}
• Subsample: {self.model.best_params['subsample']:.4f}
• Colsample: {self.model.best_params['colsample_bylevel']:.4f}
• Min Data Leaf: {self.model.best_params['min_data_in_leaf']}

TOP 5 IMPORTANT FEATURES:
"""
        for i, (_, row) in enumerate(self.model.feature_importance.head(5).iterrows(), 1):
            stats_info += f"{i}. {row['feature']}: {row['importance']:.4f}\n"
        
        stats_text.insert(1.0, stats_info)
        stats_text.config(state='disabled')
        
        # Action buttons
        action_frame = ttk.Frame(right_panel)
        action_frame.pack(fill='x', pady=(15, 0))
        
        ttk.Button(action_frame, text="💾 Save Result", 
                  command=self.save_result).pack(side='left', padx=5)
        ttk.Button(action_frame, text="📋 Copy to Clipboard", 
                  command=self.copy_to_clipboard).pack(side='left', padx=5)
        ttk.Button(action_frame, text="📤 Export Results", 
                  command=self.export_results).pack(side='left', padx=5)
        
        return tab
    
    def create_analysis_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Analysis controls
        control_frame = ttk.LabelFrame(tab, text="Visualization Tools", padding=20)
        control_frame.pack(fill='x', padx=20, pady=10)
        
        # Button grid
        button_grid = ttk.Frame(control_frame)
        button_grid.pack()
        
        analysis_buttons = [
            ("📊 Feature Importance", self.plot_feature_importance),
            ("📈 Data Distribution", self.plot_data_distribution),
            ("📉 Prediction Performance", self.plot_actual_vs_predicted),
            ("🔗 Correlation Matrix", self.plot_correlation_matrix),
            ("📋 Feature Statistics", self.plot_feature_statistics),
            ("🎯 Residual Analysis", self.plot_residuals),
        ]
        
        for i, (text, command) in enumerate(analysis_buttons):
            row, col = divmod(i, 3)
            btn = ttk.Button(button_grid, text=text, command=command, width=20)
            btn.grid(row=row, column=col, padx=5, pady=5, ipadx=10, ipady=8)
        
        # Plot area
        self.analysis_frame = ttk.Frame(tab)
        self.analysis_frame.pack(fill='both', expand=True, padx=20, pady=10)
        
        # Initial plot
        self.plot_feature_importance()
        
        return tab
    
    def create_history_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # History controls
        control_frame = ttk.LabelFrame(tab, text="Prediction History", padding=20)
        control_frame.pack(fill='x', padx=20, pady=10)
        
        # Control buttons
        button_frame = ttk.Frame(control_frame)
        button_frame.pack()
        
        ttk.Button(button_frame, text="🗑️ Clear History", 
                  command=self.clear_history).pack(side='left', padx=5)
        ttk.Button(button_frame, text="💾 Export to CSV", 
                  command=self.export_history).pack(side='left', padx=5)
        ttk.Button(button_frame, text="📤 Load History", 
                  command=self.load_history_from_file).pack(side='left', padx=5)
        ttk.Button(button_frame, text="🔄 Refresh", 
                  command=self.update_history_display).pack(side='left', padx=5)
        
        # Statistics
        self.history_stats_var = tk.StringVar(value="No predictions yet")
        stats_label = tk.Label(control_frame, 
                              textvariable=self.history_stats_var,
                              font=("Arial", 10),
                              fg=self.colors['steel_blue'])
        stats_label.pack(pady=(10, 0))
        
        # History table
        table_frame = ttk.Frame(tab)
        table_frame.pack(fill='both', expand=True, padx=20, pady=(0, 10))
        
        # Simple listbox for history
        self.history_listbox = tk.Listbox(table_frame,
                                         font=("Arial", 10),
                                         selectmode=tk.SINGLE)
        self.history_listbox.pack(fill='both', expand=True)
        
        # Add scrollbar
        scrollbar = ttk.Scrollbar(table_frame)
        scrollbar.pack(side='right', fill='y')
        self.history_listbox.config(yscrollcommand=scrollbar.set)
        scrollbar.config(command=self.history_listbox.yview)
        
        # Bind double-click to load history entry
        self.history_listbox.bind('<Double-Button-1>', self.load_history_entry)
        
        return tab
    
    def get_importance_color(self, importance):
        if importance > 0.1:
            return self.colors['danger']
        elif importance > 0.05:
            return self.colors['warning']
        else:
            return self.colors['primary']
    
    def load_preset(self, event=None):
        preset = self.preset_var.get()
        self.load_preset_by_name(preset)
    
    def load_preset_by_name(self, preset_name):
        # Define presets based on common concrete mixtures
        presets = {
            "Standard Mix": {
                "description": "Standard concrete mix (moderate strength)",
                "multiplier": 1.0
            },
            "High Strength Mix": {
                "description": "High strength concrete mix (high C-S)",
                "multiplier": 1.3
            },
            "Low Strength Mix": {
                "description": "Low strength concrete mix (low C-S)",
                "multiplier": 0.7
            },
            "Optimal Mix": {
                "description": "Optimized mix design",
                "multiplier": 1.2
            }
        }
        
        if preset_name in presets:
            multiplier = presets[preset_name]["multiplier"]
            
            for feature in self.entries:
                stats = self.feature_stats[feature]
                # Adjust based on feature importance
                importance = self.model.feature_importance[
                    self.model.feature_importance['feature'] == feature
                ]['importance'].values[0] if feature in self.model.feature_importance['feature'].values else 0
                
                # Important features get bigger adjustment
                adj_multiplier = multiplier * (1 + importance * 0.5)
                new_value = stats['mean'] * adj_multiplier
                
                # Ensure within bounds
                new_value = max(stats['min'], min(stats['max'], new_value))
                
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{new_value:.2f}")
            
            messagebox.showinfo("Preset Loaded", 
                               f"Loaded '{preset_name}' preset\n{presets[preset_name]['description']}")
    
    def fill_random_sample(self):
        """Fill with a random sample from dataset"""
        random_idx = np.random.randint(0, len(self.df))
        sample = self.df.iloc[random_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                value = sample[feature]
                self.entries[feature].insert(0, f"{value:.2f}")
        
        actual_strength = sample[self.target_name]
        messagebox.showinfo("Random Sample", 
                           f"Loaded random sample #{random_idx}\nActual C-S: {actual_strength:.1f} MPa")
    
    def fill_best_case(self):
        """Fill with values that give highest predicted strength"""
        # Use the sample with highest actual strength
        best_idx = self.y.idxmax()
        sample = self.df.iloc[best_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{sample[feature]:.2f}")
        
        messagebox.showinfo("Best Case", 
                           f"Loaded best case scenario\nMaximum C-S in dataset: {self.y.max():.1f} MPa")
    
    def fill_worst_case(self):
        """Fill with values that give lowest predicted strength"""
        # Use the sample with lowest actual strength
        worst_idx = self.y.idxmin()
        sample = self.df.iloc[worst_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{sample[feature]:.2f}")
        
        messagebox.showinfo("Worst Case", 
                           f"Loaded worst case scenario\nMinimum C-S in dataset: {self.y.min():.1f} MPa")
    
    def update_entry_from_slider(self, feature, value):
        """Update entry field when slider moves"""
        if feature in self.entries:
            self.entries[feature].delete(0, tk.END)
            self.entries[feature].insert(0, f"{value:.2f}")
    
    def threaded_predict(self):
        """Run prediction in separate thread"""
        self.predict_btn.config(state='disabled', text="Predicting...")
        thread = threading.Thread(target=self.predict_strength)
        thread.daemon = True
        thread.start()
    
    # ========== FIXED PREDICTION METHODS WITH CORRECT SCALING ==========
    
    def predict_strength(self):
        """Predict concrete strength with properly scaled SHAP contributions"""
        try:
            # Get input values
            inputs = {}
            for feature, entry in self.entries.items():
                value = entry.get().strip()
                if value:
                    try:
                        inputs[feature] = float(value)
                    except:
                        inputs[feature] = self.feature_stats[feature]['mean']
                else:
                    inputs[feature] = self.feature_stats[feature]['mean']
            
            # Create input dataframe
            input_df = pd.DataFrame([inputs], columns=self.feature_names)
            
            # IMPORTANT FIX: Scale input exactly as used for prediction
            input_scaled = self.model.scaler.transform(input_df)
            
            # Make prediction using scaled inputs
            prediction = self.model.model.predict(input_scaled)[0]
            
            # Calculate SHAP values using the SAME scaled input
            feature_contributions = {}
            shap_prediction = None
            
            try:
                # Get SHAP values from CatBoost using the scaled input
                # This returns array: [feature_contributions..., base_value]
                shap_values = self.model.model.get_feature_importance(
                    input_scaled,  # Use scaled input, NOT input_df
                    type='ShapValues', 
                    thread_count=-1
                )[0]
                
                # The last element is the base value (expected value)
                base_value = shap_values[-1]
                
                # Extract feature contributions for each feature
                for i, feature in enumerate(self.feature_names):
                    if i < len(shap_values) - 1:
                        feature_contributions[feature] = shap_values[i]
                
                # Calculate SHAP reconstruction
                shap_prediction = base_value + sum(feature_contributions.values())
                
                # Verify SHAP values match the prediction
                if np.isclose(shap_prediction, prediction, rtol=1e-5, atol=1e-5):
                    print(f"✓ SHAP verified: {base_value:.6f} + {sum(feature_contributions.values()):.6f} = {prediction:.6f}")
                else:
                    print(f"⚠️ SHAP verification: SHAP={shap_prediction:.8f}, Prediction={prediction:.8f}, Diff={shap_prediction - prediction:.8e}")
                    
            except Exception as shap_error:
                # Fallback: Calculate approximate contributions using feature importance
                print(f"SHAP calculation failed, using fallback method: {shap_error}")
                
                # Get feature importance weights
                importance_df = self.model.feature_importance
                total_importance = importance_df['importance'].sum()
                
                # Calculate normalized contributions
                total_contrib = 0
                for feature in self.feature_names:
                    if feature in inputs:
                        # Get feature statistics from training data
                        feature_mean = self.X[feature].mean()
                        feature_std = self.X[feature].std()
                        
                        # Get importance weight
                        importance_val = importance_df[
                            importance_df['feature'] == feature
                        ]['importance'].values[0] if feature in importance_df['feature'].values else 0
                        
                        # Calculate normalized contribution
                        if feature_std > 0:
                            z_score = (inputs[feature] - feature_mean) / feature_std
                            normalized_contrib = z_score * importance_val
                        else:
                            normalized_contrib = 0
                        
                        feature_contributions[feature] = normalized_contrib
                        total_contrib += normalized_contrib
                
                # Scale contributions to match prediction
                if abs(total_contrib) > 1e-10:
                    scale_factor = prediction / total_contrib
                    for feature in feature_contributions:
                        feature_contributions[feature] *= scale_factor
                else:
                    # If total contribution is zero, distribute prediction evenly
                    for feature in feature_contributions:
                        feature_contributions[feature] = prediction / len(feature_contributions)
                
                # Set base value for display
                base_value = 0
                shap_prediction = prediction
            
            # Store base value for display
            self.last_base_value = base_value if 'base_value' in locals() else 0
            self.last_shap_prediction = shap_prediction if shap_prediction is not None else prediction
            
            # Update GUI with results
            self.root.after(0, lambda: self.display_result(
                prediction, inputs, feature_contributions))
            
        except Exception as e:
            error_msg = f"Prediction failed: {str(e)}"
            print(f"❌ {error_msg}")
            import traceback
            traceback.print_exc()
            self.root.after(0, lambda: messagebox.showerror("Prediction Error", error_msg))
        
        self.root.after(0, self.enable_predict_button)
    
    def display_result(self, prediction, inputs, feature_contributions):
        """Display prediction results with properly calculated contributions"""
        # Format result
        result_text = f"{prediction:.1f} MPa"
        self.result_var.set(result_text)
        
        # Update feature contributions with correct units
        self.update_contributions_text(feature_contributions, prediction)
        
        # Save to history
        self.save_to_history(prediction, inputs)
        
        # Success message
        self.show_success_animation()
    
    def update_contributions_text(self, contributions, prediction):
        """Update the contributions text with properly formatted values in MPa units"""
        self.contributions_text.config(state='normal')
        self.contributions_text.delete(1.0, tk.END)
        
        if contributions:
            # Create header
            text = "FEATURE CONTRIBUTIONS (MPa units)\n"
            text += "=" * 50 + "\n\n"
            
            # Sort by absolute contribution value
            sorted_contrib = sorted(contributions.items(), key=lambda x: abs(x[1]), reverse=True)
            
            # Calculate total contribution and base value
            total_contrib = sum(contributions.values())
            
            # Use stored base value if available, otherwise estimate from prediction
            if hasattr(self, 'last_base_value'):
                base_value = self.last_base_value
            else:
                base_value = prediction - total_contrib
            
            # Display individual contributions
            for feature, contribution in sorted_contrib:
                if abs(contribution) > 0.001:  # Only show meaningful contributions
                    direction = "↑ Increases" if contribution > 0 else "↓ Decreases"
                    percentage = (abs(contribution) / abs(prediction) * 100) if abs(prediction) > 0.001 else 0
                    
                    # Format contribution with +/- sign
                    contrib_str = f"{contribution:+8.2f} MPa"
                    pct_str = f"({percentage:.1f}%)"
                    
                    text += f"• {feature[:25]:25} {direction:12} {contrib_str} {pct_str}\n"
            
            # Add summary
            text += "\n" + "=" * 50 + "\n"
            text += f"Base Value:      {base_value:8.2f} MPa\n"
            text += f"Net Change:     {total_contrib:+8.2f} MPa\n"
            text += f"Final Prediction: {prediction:8.1f} MPa\n"
            
            # Verification
            verification = base_value + total_contrib
            if abs(verification - prediction) < 0.1:
                text += "\n✓ Contributions sum correctly to prediction"
            else:
                text += f"\n⚠️ Note: Contributions sum to {verification:.2f} MPa (expected {prediction:.2f} MPa)"
            
            # Add interpretation
            text += "\n\n" + "=" * 50 + "\n"
            text += "INTERPRETATION:\n"
            
            positive_contribs = [c for c in contributions.values() if c > 0]
            negative_contribs = [c for c in contributions.values() if c < 0]
            
            if positive_contribs:
                top_positive = max(contributions.items(), key=lambda x: x[1])
                text += f"• {top_positive[0]} is the strongest contributor to higher strength "
                text += f"({top_positive[1]:.2f} MPa increase)\n"
            
            if negative_contribs:
                top_negative = min(contributions.items(), key=lambda x: x[1])
                text += f"• {top_negative[0]} is the strongest contributor to lower strength "
                text += f"({top_negative[1]:.2f} MPa decrease)\n"
            
            # Add context
            if prediction > 50:
                text += "• This is considered HIGH strength concrete (> 50 MPa)"
            elif prediction > 30:
                text += "• This is considered MEDIUM strength concrete (30-50 MPa)"
            else:
                text += "• This is considered LOW strength concrete (< 30 MPa)"
            
            self.contributions_text.insert(1.0, text)
        else:
            self.contributions_text.insert(1.0, "No contribution data available.")
        
        self.contributions_text.config(state='disabled')
    
    def enable_predict_button(self):
        self.predict_btn.config(state='normal', text="🔮 PREDICT C-S (MPa)")
    
    def show_success_animation(self):
        original_bg = self.predict_btn.cget('style')
        self.predict_btn.configure(style='Success.TButton')
        self.root.after(1000, lambda: self.predict_btn.configure(style=original_bg))
    
    def save_to_history(self, prediction, inputs):
        history_entry = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'c_s_mpa': float(prediction),
            **{k: inputs[k] for k in self.model.feature_importance.head(5)['feature'].tolist() if k in inputs}
        }
        
        self.prediction_history.append(history_entry)
        self.update_history_display()
        self.save_history()
    
    def update_history_display(self):
        self.history_listbox.delete(0, tk.END)
        
        for entry in self.prediction_history[-50:]:  # Last 50 entries
            display_text = f"{entry['timestamp']} - {entry['c_s_mpa']:.1f} MPa"
            self.history_listbox.insert(tk.END, display_text)
        
        if self.prediction_history:
            strengths = [h['c_s_mpa'] for h in self.prediction_history]
            stats_text = (f"Total Predictions: {len(self.prediction_history)} | "
                         f"Average: {np.mean(strengths):.1f} MPa | "
                         f"Range: {min(strengths):.1f}-{max(strengths):.1f} MPa")
            self.history_stats_var.set(stats_text)
    
    def load_history_entry(self, event):
        selection = self.history_listbox.curselection()
        if selection:
            idx = selection[0]
            if idx < len(self.prediction_history):
                entry = self.prediction_history[idx]
                
                # Fill inputs with historical values
                for feature in self.entries:
                    if feature in entry:
                        self.entries[feature].delete(0, tk.END)
                        self.entries[feature].insert(0, f"{entry[feature]:.2f}")
                
                # Update result display
                self.result_var.set(f"{entry['c_s_mpa']:.1f} MPa")
                
                messagebox.showinfo("History Loaded", 
                                   f"Loaded prediction from {entry['timestamp']}")
    
    def clear_inputs(self):
        for feature, entry in self.entries.items():
            entry.delete(0, tk.END)
            entry.insert(0, f"{self.feature_stats[feature]['mean']:.2f}")
        
        self.result_var.set("Ready for prediction")
        self.contributions_text.config(state='normal')
        self.contributions_text.delete(1.0, tk.END)
        self.contributions_text.insert(1.0, "Feature contributions will appear here after prediction.")
        self.contributions_text.config(state='disabled')
    
    def save_result(self):
        """Save current prediction result"""
        result_text = f"""CONCRETE C-S (MPa) PREDICTION - CatBoost + MOWCA
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Predicted C-S: {self.result_var.get()}

Input Parameters:
"""
        
        for feature in self.entries:
            result_text += f"{feature}: {self.entries[feature].get()}\n"
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")],
            initialfile=f"concrete_cs_prediction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        )
        
        if filename:
            try:
                with open(filename, 'w') as f:
                    f.write(result_text)
                messagebox.showinfo("Save Successful", f"Result saved to {filename}")
            except Exception as e:
                messagebox.showerror("Save Failed", f"Failed to save: {str(e)}")
    
    def copy_to_clipboard(self):
        """Copy result to clipboard"""
        result_text = f"Predicted Concrete C-S: {self.result_var.get()}"
        self.root.clipboard_clear()
        self.root.clipboard_append(result_text)
        messagebox.showinfo("Copied", "Result copied to clipboard!")
    
    def export_results(self):
        """Export detailed results"""
        self.export_history()
    
    def clear_history(self):
        if not self.prediction_history:
            messagebox.showinfo("No History", "Prediction history is already empty.")
            return
        
        if messagebox.askyesno("Clear History", "Clear all prediction history?"):
            self.prediction_history = []
            self.update_history_display()
            self.save_history()
            messagebox.showinfo("History Cleared", "Prediction history cleared.")
    
    def export_history(self):
        if not self.prediction_history:
            messagebox.showwarning("No Data", "No prediction history to export.")
            return
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx"), ("All files", "*.*")],
            initialfile=f"concrete_cs_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        )
        
        if filename:
            try:
                df = pd.DataFrame(self.prediction_history)
                if filename.endswith('.xlsx'):
                    df.to_excel(filename, index=False)
                else:
                    df.to_csv(filename, index=False)
                messagebox.showinfo("Export Successful", f"History exported to {filename}")
            except Exception as e:
                messagebox.showerror("Export Failed", f"Failed to export: {str(e)}")
    
    def load_history_from_file(self):
        filename = filedialog.askopenfilename(
            filetypes=[("JSON files", "*.json"), ("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if filename:
            try:
                if filename.endswith('.json'):
                    with open(filename, 'r') as f:
                        self.prediction_history = json.load(f)
                elif filename.endswith('.csv'):
                    self.prediction_history = pd.read_csv(filename).to_dict('records')
                
                self.update_history_display()
                messagebox.showinfo("Load Successful", 
                                   f"Loaded {len(self.prediction_history)} predictions")
            except Exception as e:
                messagebox.showerror("Load Failed", f"Failed to load: {str(e)}")
    
    def save_history(self):
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.prediction_history, f, indent=2)
        except Exception as e:
            print(f"Failed to save history: {e}")
    
    def load_history(self):
        try:
            if os.path.exists(self.history_file):
                with open(self.history_file, 'r') as f:
                    self.prediction_history = json.load(f)
                self.update_history_display()
        except:
            pass
    
    # ========== PLOTTING FUNCTIONS ==========
    
    def plot_feature_importance(self):
        self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 8), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        # Plot all features
        importance_df = self.model.feature_importance
        y_pos = np.arange(len(importance_df))
        
        colors = cm.viridis(np.linspace(0.3, 0.9, len(importance_df)))
        bars = ax.barh(y_pos, importance_df['importance'], 
                      color=colors, edgecolor='black', height=0.7)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(importance_df['feature'], fontsize=10)
        ax.set_xlabel('Feature Importance Score', fontsize=12, fontweight='bold')
        ax.set_title('CatBoost Model - Feature Importance for C-S Prediction', 
                    fontsize=14, fontweight='bold', pad=20)
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')
        
        # Add value labels
        for i, (bar, importance) in enumerate(zip(bars, importance_df['importance'])):
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                   f'{importance:.3f}', va='center', fontsize=9, fontweight='bold')
        
        fig.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        widget = canvas.get_tk_widget()
        widget.pack(fill='both', expand=True)
        
        self.current_plot = fig
    
    def plot_data_distribution(self):
        self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 8), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        # Create histogram
        n, bins, patches = ax.hist(self.y, bins=30, alpha=0.7, 
                                  color=self.colors['steel_blue'], 
                                  edgecolor='black', density=True)
        
        ax.set_xlabel('C-S (MPa)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Density', fontsize=12, fontweight='bold')
        ax.set_title('Distribution of Concrete C-S (MPa)', 
                    fontsize=14, fontweight='bold', pad=20)
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # Add statistics box
        stats_text = (f'Mean: {self.y.mean():.1f} MPa\n'
                     f'Std: {self.y.std():.1f} MPa\n'
                     f'Min: {self.y.min():.1f} MPa\n'
                     f'Max: {self.y.max():.1f} MPa\n'
                     f'Samples: {len(self.y)}')
        
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
               fontsize=10, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        fig.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        widget = canvas.get_tk_widget()
        widget.pack(fill='both', expand=True)
        
        self.current_plot = fig
    
    def plot_actual_vs_predicted(self):
        self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 10), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        # Get predictions
        X_scaled = self.model.scaler.transform(self.X)
        predictions = self.model.model.predict(X_scaled)
        
        scatter = ax.scatter(self.y, predictions, 
                            alpha=0.6, 
                            s=50, 
                            c=self.y, 
                            cmap='viridis',
                            edgecolors='white', 
                            linewidth=0.5)
        
        min_val = min(self.y.min(), predictions.min())
        max_val = max(self.y.max(), predictions.max())
        ax.plot([min_val, max_val], [min_val, max_val], 
               'r--', linewidth=2, alpha=0.7, label='Perfect Prediction')
        
        ax.set_xlabel('Actual C-S (MPa)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Predicted C-S (MPa)', fontsize=12, fontweight='bold')
        ax.set_title('CatBoost Model: Actual vs Predicted C-S', 
                    fontsize=14, fontweight='bold', pad=20)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=10)
        
        # Add R² text
        ax.text(0.05, 0.95, f'R² = {self.model.r2_score:.4f}\nRMSE = {self.model.rmse:.2f} MPa', 
               transform=ax.transAxes, fontsize=10,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label('Actual C-S (MPa)', fontsize=10)
        
        fig.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        widget = canvas.get_tk_widget()
        widget.pack(fill='both', expand=True)
        
        self.current_plot = fig
    
    def plot_correlation_matrix(self):
        self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 10), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        # Calculate correlation for all features
        corr_matrix = self.df.corr()
        
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
        
        ax.set_xticks(np.arange(len(corr_matrix.columns)))
        ax.set_yticks(np.arange(len(corr_matrix.columns)))
        ax.set_xticklabels([f[:12] for f in corr_matrix.columns], rotation=45, ha='right', fontsize=9)
        ax.set_yticklabels([f[:12] for f in corr_matrix.columns], fontsize=9)
        
        ax.set_title('Correlation Matrix - All Features', fontsize=14, fontweight='bold', pad=20)
        
        cbar = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel('Correlation Coefficient', rotation=-90, va="bottom", fontsize=10)
        
        fig.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        widget = canvas.get_tk_widget()
        widget.pack(fill='both', expand=True)
        
        self.current_plot = fig
    
    def plot_feature_statistics(self):
        self.clear_analysis_frame()
        
        fig = Figure(figsize=(14, 10), dpi=100, facecolor='white')
        
        # Plot statistics for top 6 features
        top_features = self.model.feature_importance.head(6)['feature'].tolist()
        
        for i, feature in enumerate(top_features, 1):
            ax = fig.add_subplot(2, 3, i)
            
            data = self.X[feature].dropna()
            bp = ax.boxplot(data, vert=True, patch_artist=True, 
                           widths=0.6, showfliers=True)
            
            bp['boxes'][0].set_facecolor(self.colors['primary'])
            bp['boxes'][0].set_alpha(0.7)
            
            ax.set_ylabel('Value', fontsize=9, fontweight='bold')
            ax.set_title(f'{feature}', fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        
        fig.suptitle('Statistical Distribution of Top 6 Features', 
                    fontsize=16, fontweight='bold', y=0.98)
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        widget = canvas.get_tk_widget()
        widget.pack(fill='both', expand=True)
        
        self.current_plot = fig
    
    def plot_residuals(self):
        self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 10), dpi=100, facecolor='white')
        
        X_scaled = self.model.scaler.transform(self.X)
        predictions = self.model.model.predict(X_scaled)
        residuals = self.y - predictions
        
        # Residuals vs Predicted
        ax1 = fig.add_subplot(2, 2, 1)
        ax1.scatter(predictions, residuals, alpha=0.6, s=30, edgecolors='black', linewidth=0.5)
        ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
        ax1.set_xlabel('Predicted C-S (MPa)', fontsize=10)
        ax1.set_ylabel('Residuals (MPa)', fontsize=10)
        ax1.set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # Histogram of residuals
        ax2 = fig.add_subplot(2, 2, 2)
        ax2.hist(residuals, bins=30, alpha=0.7, color=self.colors['steel_blue'], edgecolor='black')
        ax2.set_xlabel('Residuals (MPa)', fontsize=10)
        ax2.set_ylabel('Frequency', fontsize=10)
        ax2.set_title('Residual Distribution', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        # Q-Q plot
        ax3 = fig.add_subplot(2, 2, 3)
        from scipy import stats
        stats.probplot(residuals, dist="norm", plot=ax3)
        ax3.set_title('Q-Q Plot', fontsize=12, fontweight='bold')
        ax3.grid(True, alpha=0.3)
        
        # Residual statistics
        ax4 = fig.add_subplot(2, 2, 4)
        ax4.axis('off')
        stats_text = f"""Residual Statistics:
• Mean: {np.mean(residuals):.4f} MPa
• Std: {np.std(residuals):.4f} MPa
• Skewness: {stats.skew(residuals):.4f}
• Kurtosis: {stats.kurtosis(residuals):.4f}
• Min: {np.min(residuals):.4f} MPa
• Max: {np.max(residuals):.4f} MPa
• 95% Range: [{np.percentile(residuals, 2.5):.4f}, {np.percentile(residuals, 97.5):.4f}] MPa"""
        
        ax4.text(0.1, 0.5, stats_text, transform=ax4.transAxes, fontsize=10,
                verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        fig.suptitle('Residual Analysis - CatBoost Model', fontsize=14, fontweight='bold')
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        widget = canvas.get_tk_widget()
        widget.pack(fill='both', expand=True)
        
        self.current_plot = fig
    
    def clear_analysis_frame(self):
        for widget in self.analysis_frame.winfo_children():
            widget.destroy()
    
    def run(self):
        # Configure ttk styles
        style = ttk.Style()
        style.theme_use('clam')
        
        style.configure('Accent.TButton',
                       background=self.colors['accent'],
                       foreground='white',
                       font=('Arial', 11, 'bold'),
                       borderwidth=2)
        
        style.configure('Success.TButton',
                       background=self.colors['success'],
                       foreground='white',
                       font=('Arial', 11, 'bold'))
        
        print("\n" + "="*60)
        print("🚀 Starting Concrete C-S (MPa) Predictor - CatBoost + MOWCA")
        print("="*60)
        print(f"📊 Target: {self.target_name}")
        print(f"🔢 Features: {len(self.feature_names)}")
        print(f"📈 Model R²: {self.model.r2_score:.4f}")
        print(f"🔧 CatBoost Hyperparameters (MOWCA + 5-fold CV):")
        for key, value in self.model.best_params.items():
            print(f"   • {key}: {value}")
        print("="*60)
        print("✅ Application ready!")
        
        # Auto-load a preset
        self.root.after(100, lambda: self.load_preset_by_name("Standard Mix"))
        
        self.root.mainloop()

# Run the application
if __name__ == "__main__":
    try:
        print("Initializing application...")
        app = ConcreteStrengthPredictor()
        app.run()
    except Exception as e:
        print(f"❌ Application Error: {str(e)}")
        import traceback
        traceback.print_exc()
        messagebox.showerror("Application Error", f"Failed to start:\n\n{str(e)}")

Initializing application...
📂 Loading data from: D:\2026 Work\My Papers\GGSB\Data\1-s2.0-S2214509525006989-mmc1.csv
✅ Successfully loaded with encoding: latin1
✅ Dataset loaded successfully!
📐 Dataset shape: (796, 10)
📋 Column names: ['Cement (Kg/m³)', 'Water (Kg/m³)', 'W/C ', 'W/Binder', 'Sand (Kg/m³)', 'Gravel (Kg/m³)', 'Superplasticizer (kg/m³)', ' GGBS Kg/m³', 'Age (days)', 'C-S (Mpa)']
🎯 Target column: 'C-S (Mpa)'
🔍 Features loaded: 9
📊 Features: Cement_Kg_m³, Water_Kg_m³, W_C, W_Binder, Sand_Kg_m³, Gravel_Kg_m³, Superplasticizer_kg_m³, GGBS_Kg_m³, Age_days

📊 Target Statistics (C-S (Mpa)):
   • Mean: 42.39 MPa
   • Std: 20.90 MPa
   • Min: 2.33 MPa
   • Max: 101.30 MPa

🤖 Optimizing CatBoost with MOWCA + 5-fold CV...
⏳ Running MOWCA optimization (this may take 2-5 minutes)...
Iteration 0: Best Fitness = -0.9769
